### Initialisation

In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

### Load Data

In [34]:
# Load the training data
train_df = pd.read_csv('../data/application_train.csv')
print(f"Training data shape: {train_df.shape}")
print(f"Default rate: {train_df['TARGET'].mean():.2%}")
print(f"Missing value %: {(train_df.isnull().sum().sum() / train_df.size) * 100:.1f}%")

Training data shape: (307511, 122)
Default rate: 8.07%
Missing value %: 24.4%


### Core Features

In [35]:
# Core features
train_df['CREDIT_INCOME_RATIO'] = train_df['AMT_CREDIT'] / train_df['AMT_INCOME_TOTAL'].replace(0, np.nan)
train_df['ANNUITY_INCOME_RATIO'] = train_df['AMT_ANNUITY'] / train_df['AMT_INCOME_TOTAL'].replace(0, np.nan)

# Days to years
train_df['DAYS_BIRTH_YEARS'] = -train_df['DAYS_BIRTH'] / 365
train_df['DAYS_EMPLOYED_YEARS'] = train_df['DAYS_EMPLOYED'].apply(lambda x: -x/365 if x < 0 else np.nan)

# Feature: ratio of employment to age
train_df['EMPLOYMENT_TO_AGE_RATIO'] = train_df['DAYS_EMPLOYED_YEARS'] / train_df['DAYS_BIRTH_YEARS']

# Feature: family size feature
train_df['FAMILY_SIZE'] = train_df['CNT_FAM_MEMBERS'].fillna(0) + 1

# Feature: thin file flag (if all 3 external sources missing -> flag as thin file)
train_df['THIN_FILE_FLAG'] = train_df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].isnull().all(axis=1).astype(int)

print(f"Shape after new features added: {train_df.shape}")

Shape after new features added: (307511, 129)


In [36]:
# Save the processed training data for modelling
train_df.to_csv('../outputs/data/train_data.csv', index=False)

### Segment Analysis

In [37]:
# Segment 1: High debt burden
high_debt = train_df['CREDIT_INCOME_RATIO'] > 3.5
default_high_debt = train_df[high_debt]['TARGET'].mean()
default_low_debt = train_df[~high_debt]['TARGET'].mean()
print(f"\n  High debt burden (>3.5x): {default_high_debt:.2%} default rate")
print(f"  Low debt burden: {default_low_debt:.2%} default rate")
print(f"  Risk multiplier: {default_high_debt/default_low_debt:.1f}x")

# Segment 2: Thin-file applicants
thin_default = train_df[train_df['THIN_FILE_FLAG'] == 1]['TARGET'].mean()
thick_default = train_df[train_df['THIN_FILE_FLAG'] == 0]['TARGET'].mean()
print(f"\n  Thin-file applicants: {thin_default:.2%} default rate")
print(f"  With bureau history: {thick_default:.2%} default rate")
print(f"  Risk multiplier: {thin_default/thick_default:.1f}x")

# Segment 3: External score analysis 
if 'EXT_SOURCE_2' in train_df.columns:
    low_score = train_df['EXT_SOURCE_2'] < 0.3
    high_score = train_df['EXT_SOURCE_2'] > 0.6
    if low_score.any() and high_score.any():
        print(f"\n  Low EXT_SOURCE_2 (<0.3): {train_df[low_score]['TARGET'].mean():.2%} default")
        print(f"  High EXT_SOURCE_2 (>0.6): {train_df[high_score]['TARGET'].mean():.2%} default")

# Segment 4: Employment stability
short_emp = train_df['DAYS_EMPLOYED_YEARS'] < 1
long_emp = train_df['DAYS_EMPLOYED_YEARS'] > 5
if short_emp.any() and long_emp.any():
    print(f"\n  Employed <1 year: {train_df[short_emp]['TARGET'].mean():.2%} default")
    print(f"  Employed >5 years: {train_df[long_emp]['TARGET'].mean():.2%} default")


  High debt burden (>3.5x): 7.95% default rate
  Low debt burden: 8.18% default rate
  Risk multiplier: 1.0x

  Thin-file applicants: 8.14% default rate
  With bureau history: 8.07% default rate
  Risk multiplier: 1.0x

  Low EXT_SOURCE_2 (<0.3): 15.88% default
  High EXT_SOURCE_2 (>0.6): 4.56% default

  Employed <1 year: 10.98% default
  Employed >5 years: 6.41% default


### Modelling Preparation

In [38]:
# Select features (exclude target and identifiers)
exclude_cols = ['TARGET', 'SK_ID_CURR']
feature_cols = [col for col in train_df.columns if col not in exclude_cols]

# Handle missing values - simple median imputation for now
X = train_df[feature_cols].copy()
y = train_df['TARGET'].copy()

# Convert object types
for col in X.select_dtypes(include=['object']).columns:
    X[col] = X[col].astype('category')

# Median impute numeric
numeric_cols = X.select_dtypes(include=[np.number]).columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Class balance: {y_train.mean():.2%} default in training")

Training set: (246008, 127)
Validation set: (61503, 127)
Class balance: 8.07% default in training


### Baseline Model

In [39]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Simple Random Forest (faster than XGBoost for initial test)
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train.select_dtypes(include=[np.number]), y_train)

# Predict
y_pred_proba = rf.predict_proba(X_val.select_dtypes(include=[np.number]))[:, 1]
auc = roc_auc_score(y_val, y_pred_proba)
print(f"Baseline Random Forest ROC-AUC: {auc:.4f}")

Baseline Random Forest ROC-AUC: 0.7010


### Save Initial Results

In [40]:
# Create summary dataframe for dashboard
segment_summary = pd.DataFrame({
    'Segment': ['High Debt Burden (>3.5x)', 'Low Debt Burden (≤3.5x)', 
                'Thin-File Applicants', 'With Bureau History',
                'Employed <1 Year', 'Employed >5 Years'],
    'Default_Rate': [default_high_debt, default_low_debt, 
                     thin_default, thick_default,
                     train_df[short_emp]['TARGET'].mean() if short_emp.any() else 0,
                     train_df[long_emp]['TARGET'].mean() if long_emp.any() else 0],
    'Sample_Size': [high_debt.sum(), (~high_debt).sum(),
                    train_df[train_df['THIN_FILE_FLAG']==1].shape[0],
                    train_df[train_df['THIN_FILE_FLAG']==0].shape[0],
                    short_emp.sum() if short_emp.any() else 0,
                    long_emp.sum() if long_emp.any() else 0]
})

segment_summary.to_csv('../outputs/segment_analysis.csv', index=False)
print("Saved segment_analysis.csv for dashboard")

# Save model
import joblib
joblib.dump(rf, '../outputs/models/baseline_model.pkl')
print("Saved baseline_model.pkl")

print("\n" + "=" * 60)
print("EDA + Feature Engineering complete!")
print("Next: Run modelling notebook to improve ROC-AUC")
print("=" * 60)

# Display key metrics for your consulting summary
print("\nKEY METRICS SUMMARY:")
print("-" * 40)
print(f"ROC-AUC (baseline): {auc:.3f}")
print(f"High-debt risk multiplier: {default_high_debt/default_low_debt:.1f}x")
print(f"Thin-file risk multiplier: {thin_default/thick_default:.1f}x")

Saved segment_analysis.csv for dashboard
Saved baseline_model.pkl

EDA + Feature Engineering complete!
Next: Run modelling notebook to improve ROC-AUC

KEY METRICS SUMMARY:
----------------------------------------
ROC-AUC (baseline): 0.701
High-debt risk multiplier: 1.0x
Thin-file risk multiplier: 1.0x
